In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
import re

# --- Step 0: 环境设置  ---
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# 绝对路径 
PROJECT_FOLDER_PATH = r"C:\Fama_French_Project"
DATA_FOLDER = os.path.join(PROJECT_FOLDER_PATH, 'data')

PATH_RF = os.path.join(DATA_FOLDER, 'risk_free.csv')
PATH_BOOK = os.path.join(DATA_FOLDER, 'balance_sheet.csv')
PATH_RET = os.path.join(DATA_FOLDER, 'ret_monthly.csv')
OUTPUT_PATH = os.path.join(DATA_FOLDER, 'cleaned_ff_data.csv')

print(f"工作目录: {DATA_FOLDER}")

# --- 工具函数 ---
def read_csmr_safe(path):
    """尝试多种编码读取"""
    for enc in ['utf-8-sig', 'gbk', 'utf-8', 'gb18030']:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            continue
    raise ValueError(f"无法读取文件: {path}")

def clean_code(series):
    """核弹级代码清洗：去空格、去后缀、补零"""
    return series.astype(str).apply(lambda x: re.sub(r'\D', '', x)).str.zfill(6)

# --- Step 1: 处理无风险利率 (RF)  ---
print("\n[1/4] 正在处理无风险利率...")
try:
    df_rf = read_csmr_safe(PATH_RF)
    

    rate_col = 'Nrrmtdt'  
    date_col = 'Clsdt'
    
    print(f"   - 锁定列: 利率=[{rate_col}], 日期=[{date_col}]")
    
    # 强制转数字
    df_rf['rf'] = pd.to_numeric(df_rf[rate_col], errors='coerce')
    df_rf.dropna(subset=['rf'], inplace=True)
    
    # 单位修正
    # 截图显示数值是 0.1241。
    # 中国的月度无风险利率通常在 0.1% - 0.2% 左右 (年化 1.5% - 2.5%)。
    # 所以 0.1241 绝对是“百分数” (0.1241%)。
    # 我们只需要除以 100 变成小数。不需要再除以 12 了（因为它已经是月度值）。
    print(f"   - 原始值示例: {df_rf['rf'].iloc[0]} (判断为百分数)")
    df_rf['rf'] = df_rf['rf'] / 100 

    # 时间索引
    df_rf['trd_month'] = pd.to_datetime(df_rf[date_col]).dt.to_period('M')
    
    # 聚合（取月均值，防止一个月有多条）
    df_rf = df_rf[['trd_month', 'rf']].groupby('trd_month')['rf'].mean().reset_index()
    
    print(f"   - RF 处理完毕，共 {len(df_rf)} 行")
    print(f"   - RF 均值: {df_rf['rf'].mean():.6f}")

except Exception as e:
    print(f"【错误】无风险利率处理失败: {e}")
    raise

# --- Step 2: 处理财务数据 (Book) ---
print("\n[2/4] 正在处理资产负债表...")
df_book = read_csmr_safe(PATH_BOOK)

# 筛选
df_book['date'] = pd.to_datetime(df_book['Accper'])
df_book = df_book[(df_book['Typrep'] == 'A') & (df_book['date'].dt.month == 12)]

# 核心清洗
df_book['code'] = clean_code(df_book['Stkcd'])
df_book['year'] = df_book['date'].dt.year.astype(int)
df_book['match_year'] = (df_book['year'] + 1).astype(int) # 滞后匹配
df_book['BookValue'] = pd.to_numeric(df_book['A003000000'], errors='coerce')

df_book = df_book[['code', 'match_year', 'BookValue']].dropna()
print(f"   - Book 处理完毕，共 {len(df_book)} 行")


# --- Step 3: 处理回报数据 (Ret) ---
print("\n[3/4] 正在处理月度回报率...")
df_ret = read_csmr_safe(PATH_RET)

# 核心清洗
df_ret['code'] = clean_code(df_ret['Stkcd'])
df_ret['trd_month'] = pd.to_datetime(df_ret['Trdmnt']).dt.to_period('M')
df_ret['year'] = df_ret['trd_month'].dt.year.astype(int)
df_ret['month'] = df_ret['trd_month'].dt.month.astype(int)

# 匹配年份逻辑
df_ret['match_year'] = np.where(df_ret['month'] >= 7, df_ret['year'], df_ret['year'] - 1).astype(int)

# 数值清洗
df_ret['ret'] = pd.to_numeric(df_ret['Mretwd'], errors='coerce')
df_ret['me'] = pd.to_numeric(df_ret['Msmvttl'], errors='coerce')

df_ret = df_ret[['code', 'trd_month', 'match_year', 'ret', 'me']].dropna()
print(f"   - Ret 处理完毕，共 {len(df_ret)} 行")


# --- Step 4: 最终合并 (The Grand Merge) ---
print("\n[4/4] 执行最终合并...")

# 1. 股票 + 财务
df_merge = pd.merge(df_ret, df_book, on=['code', 'match_year'], how='inner')
print(f"   - 股票与财务合并后: {len(df_merge)} 行 (如果为0，说明年份或代码没对上)")

if len(df_merge) > 0:
    # 2. + 无风险利率
    df_merge = pd.merge(df_merge, df_rf, on='trd_month', how='left')
    
    # 3. 计算 BM (自动判断单位)
    sample_me = df_merge['me'].iloc[0]
    if sample_me < 1e9: # 如果市值小于10亿，肯定是千元单位
        print("   - 检测到市值单位为千元，正在修正...")
        df_merge['me_yuan'] = df_merge['me'] * 1000
    else:
        df_merge['me_yuan'] = df_merge['me']
        
    df_merge['BM'] = df_merge['BookValue'] / df_merge['me_yuan']
    
    # 4. 剔除负资产
    df_merge = df_merge[df_merge['BM'] > 0]
    
    # 5. 最终清洗
    df_merge = df_merge.dropna(subset=['ret', 'me', 'BM', 'rf'])
    
    # 6. 计算超额收益
    df_merge['eret'] = df_merge['ret'] - df_merge['rf']
    
    # 保存
    df_merge.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    print(f"\n✅ 大功告成！最终有效数据: {len(df_merge)} 行")
    print(f"文件已保存至: {OUTPUT_PATH}")
    
    print("\n数据预览:")
    display(df_merge[['trd_month', 'code', 'ret', 'rf', 'eret', 'BM']].head())
    print("\nBM 统计:")
    display(df_merge['BM'].describe())
    
else:
    print("\n❌ 严重错误：合并结果为 0 行！")
    print("请检查：1. CSMAR下载的时间范围是否重叠？ 2. 股票代码是否还有问题？")

工作目录: C:\Fama_French_Project\data

[1/4] 正在处理无风险利率...
   - 锁定列: 利率=[Nrrmtdt], 日期=[Clsdt]
   - 原始值示例: 0.1241 (判断为百分数)
   - RF 处理完毕，共 120 行
   - RF 均值: 0.001211

[2/4] 正在处理资产负债表...
   - Book 处理完毕，共 44228 行

[3/4] 正在处理月度回报率...
   - Ret 处理完毕，共 513676 行

[4/4] 执行最终合并...
   - 股票与财务合并后: 474431 行 (如果为0，说明年份或代码没对上)
   - 检测到市值单位为千元，正在修正...

✅ 大功告成！最终有效数据: 472402 行
文件已保存至: C:\Fama_French_Project\data\cleaned_ff_data.csv

数据预览:


,trd_month,code,ret,rf,eret,BM
0,2016-07,000001,0.057471,0.001241,0.056230,1.022360
1,2016-08,000001,0.031522,0.001241,0.030281,0.991119
2,2016-09,000001,-0.044257,0.001241,-0.045498,1.037014
3,2016-10,000001,0.008820,0.001241,0.007579,1.027947
4,2016-11,000001,0.043716,0.001241,0.042475,0.984892



BM 统计:


count    472402.000000
mean          1.201736
std          10.505512
min           0.000019
25%           0.238034
50%           0.404581
75%           0.667797
max        1401.987216
Name: BM, dtype: float64

In [22]:
# --- 教授的“开颅诊断”代码 ---
# 我们只看 risk_free.csv 到底长什么样

import os

PROJECT_FOLDER_PATH = r"C:\Fama_French_Project"
PATH_RF = os.path.join(PROJECT_FOLDER_PATH, 'data', 'risk_free.csv')

print(f"正在读取: {PATH_RF}")

# 1. 直接以文本模式读取前5行，不经过 Pandas 解析
with open(PATH_RF, 'r', encoding='utf-8') as f:
    print("\n--- [UTF-8] 原始文本前5行 ---")
    for i in range(5):
        print(f"Line {i}: {f.readline().strip()}")

# 2. 尝试用 Pandas 读取并展示原始 DataFrame
import pandas as pd
try:
    df_debug = pd.read_csv(PATH_RF, encoding='utf-8-sig') # 尝试最常见的编码
    print("\n--- [Pandas] 原始表格前5行 ---")
    display(df_debug.head())
    print("\n--- [Pandas] 数据类型 ---")
    print(df_debug.dtypes)
except Exception as e:
    print(f"\nPandas 读取失败: {e}")

正在读取: C:\Fama_French_Project\data\risk_free.csv

--- [UTF-8] 原始文本前5行 ---
Line 0: ﻿"Nrr1","Clsdt","Nrrmtdt"
Line 1: "NRI01","2016-01-01","0.124100"
Line 2: "NRI01","2016-01-02","0.124100"
Line 3: "NRI01","2016-01-03","0.124100"
Line 4: "NRI01","2016-01-04","0.124100"

--- [Pandas] 原始表格前5行 ---


,Nrr1,Clsdt,Nrrmtdt
0,NRI01,2016-01-01,0.1241
1,NRI01,2016-01-02,0.1241
2,NRI01,2016-01-03,0.1241
3,NRI01,2016-01-04,0.1241
4,NRI01,2016-01-05,0.1241



--- [Pandas] 数据类型 ---
Nrr1        object
Clsdt       object
Nrrmtdt    float64
dtype: object
